# Denoising diffusion probabilistic model (mask-conditioned U-Net)

Part of **FL-YOLO-DADP** (Chen et al., *Improved Federated Learning with Differential Privacy for YOLO Detector on Phase Images*, CDSR 2025). This is one of the four generative baselines in Table 1.

The generator learns to map a 3-channel bounding-box mask (one channel per class: RBC, WBC, PLT) to a phase image. It is trained on real phase images with YOLO labels (`generator_train`), then used to generate a synthetic version of every frame of the three agents' train/valid splits.

**Cells:** *Train* → *Synthesize* (visual check on one frame) → *Generate* (writes `../data/synthetic_raw/DDPM/agent_<k>_synthesized/`). Afterwards, from the repository root:

```bash
python data_prep/build_synthetic_dataset.py --raw data/synthetic_raw/DDPM --agents data/agents --out data/synthetic/DDPM
python baselines/train_yolo.py --data data/synthetic/DDPM/data.yaml --epochs 20 --name synthetic_DDPM
```

Paths are relative to this folder (`generative_models/`):

| path | content |
|---|---|
| `../data/generator_train/{train,valid,test}` | real frames + YOLO labels used to train the generator |
| `../data/agents/agent_<k>` | per-agent shards (input of *Generate*) |
| `../data/synthetic_raw/DDPM` | output of *Generate* |
| `../runs/generative_models/DDPM` | generator checkpoints |


In [ ]:
import os
os.makedirs("../runs/generative_models/DDPM/model/20epochs", exist_ok=True)


### Train

In [ ]:
import os
import glob
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torch.optim as optim
import cv2
from PIL import Image
from tqdm import tqdm
import random

# Set random seed for reproducibility
seed = 99
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ----------------------------------
# 1. Dataset
# ----------------------------------

class YOLOConditionalDataset(Dataset):
    """
    Loads images & YOLO-format bounding-box annotations,
    creates a multi-channel mask (one channel per class).
    Returns (mask, real_image).
    """

    def __init__(self, images_dir, labels_dir, image_size=(512, 384), num_classes=3):
        super().__init__()
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.image_size = image_size
        self.num_classes = num_classes
        
        self.image_paths = sorted(glob.glob(os.path.join(self.images_dir, "*.png")))
        
        self.to_tensor = T.ToTensor()

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        filename = os.path.splitext(os.path.basename(img_path))[0]
        label_path = os.path.join(self.labels_dir, filename + ".txt")

        # Load the real image (RGB)
        image = Image.open(img_path).convert("RGB")
        w, h = image.size

        # Create bounding-box mask
        mask = np.zeros((h, w, self.num_classes), dtype=np.uint8)
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                lines = f.readlines()
            for line in lines:
                cls_id, x_c, y_c, bw, bh = line.strip().split()
                cls_id = int(cls_id)
                x_c, y_c, bw, bh = float(x_c), float(y_c), float(bw), float(bh)

                # Convert normalized coords to pixel coords
                x_center = int(x_c * w)
                y_center = int(y_c * h)
                box_w = int(bw * w)
                box_h = int(bh * h)

                x1 = max(0, x_center - box_w // 2)
                y1 = max(0, y_center - box_h // 2)
                x2 = min(w, x_center + box_w // 2)
                y2 = min(h, y_center + box_h // 2)

                mask[y1:y2, x1:x2, cls_id] = 1

        # Resize image and mask to (384, 512)
        image = image.resize(self.image_size, Image.BILINEAR)
        mask = cv2.resize(mask, self.image_size, interpolation=cv2.INTER_NEAREST)

        # Convert to Tensor
        image = self.to_tensor(image)  # shape: [3, H, W], range [0,1]
        mask = torch.from_numpy(np.transpose(mask, (2, 0, 1))).float()  # [num_classes, H, W], 0/1

        return mask, image

# ----------------------------------
# 2. Beta Schedule and Forward Diffusion
# ----------------------------------

def make_beta_schedule(start=1e-4, end=0.02, num_timesteps=1000):
    """
    Creates a linearly spaced schedule of betas.
    """
    return torch.linspace(start, end, num_timesteps)

def forward_diffusion_sample(x0, t, sqrt_alphas_cumprod, one_minus_alphas_cumprod):
    """
    Sample from q(x_t | x_0).
    x0: [B, C, H, W]
    t: [B] of time steps
    sqrt_alphas_cumprod: [T]
    one_minus_alphas_cumprod: [T]
    """
    device = x0.device
    sqrt_alpha_cumprod_t = sqrt_alphas_cumprod[t].unsqueeze(-1).unsqueeze(-1).unsqueeze(-1)
    sqrt_one_minus_alpha_cumprod_t = torch.sqrt(one_minus_alphas_cumprod[t]).unsqueeze(-1).unsqueeze(-1).unsqueeze(-1)

    noise = torch.randn_like(x0).to(device)
    x_t = sqrt_alpha_cumprod_t * x0 + sqrt_one_minus_alpha_cumprod_t * noise
    return x_t, noise

# ----------------------------------
# 3. Model: Simple Conditional UNet
# ----------------------------------

class SimpleConditionalUNet(nn.Module):
    def __init__(self, in_channels=6, base_channels=64):
        """
        in_channels = 6 => (3 for noisy image + 3 for mask)
        out_channels = 3 => predict noise for the (RGB) image
        """
        super().__init__()
        self.enc1 = self.conv_block(in_channels, base_channels)
        self.enc2 = self.conv_block(base_channels, base_channels*2, down=True)
        self.enc3 = self.conv_block(base_channels*2, base_channels*4, down=True)
        
        self.middle = self.conv_block(base_channels*4, base_channels*4, down=False)
        
        self.dec3 = self.conv_block(base_channels*8, base_channels*2, up=True)
        self.dec2 = self.conv_block(base_channels*4, base_channels, up=True)
        self.dec1 = nn.Sequential(
            nn.Conv2d(base_channels*2, base_channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(base_channels, 3, kernel_size=1),  # Output = 3 channels
        )

    def conv_block(self, in_c, out_c, down=False, up=False):
        """
        down=True -> stride=2 conv
        up=True   -> stride=2 transpose conv
        else standard conv
        """
        if down:
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 4, 2, 1),
                nn.ReLU()
            )
        elif up:
            return nn.Sequential(
                nn.ConvTranspose2d(in_c, out_c, 4, 2, 1),
                nn.ReLU()
            )
        else:
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, 1, 1),
                nn.ReLU()
            )

    def forward(self, x, mask):
        """
        x : [B, 3, H, W] (noisy image)
        mask : [B, 3, H, W]
        """
        # Condition by concatenating
        c = torch.cat([x, mask], dim=1)  # [B, 6, H, W]
        e1 = self.enc1(c)               # [B, base, H, W]
        e2 = self.enc2(e1)              # [B, base*2, H/2, W/2]
        e3 = self.enc3(e2)              # [B, base*4, H/4, W/4]
        
        m = self.middle(e3)            # [B, base*4, H/4, W/4]
        
        d3 = self.dec3(torch.cat([m, e3], dim=1))  # skip conn
        d2 = self.dec2(torch.cat([d3, e2], dim=1))
        out = self.dec1(torch.cat([d2, e1], dim=1))
        return out

# ----------------------------------
# 4. Training Loop
# ----------------------------------

def train_diffusion():
    # Paths
    train_images_dir = "../data/generator_train/train/images"
    train_labels_dir = "../data/generator_train/train/labels"
    val_images_dir = "../data/generator_train/valid/images"
    val_labels_dir = "../data/generator_train/valid/labels"
    
    # Hyperparameters
    image_size = (512, 384)
    batch_size = 4
    lr = 1e-4
    num_epochs = 20
    num_timesteps = 200  # Set to 200
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Dataset & DataLoader
    train_dataset = YOLOConditionalDataset(
        images_dir=train_images_dir,
        labels_dir=train_labels_dir,
        image_size=image_size
    )
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    
    val_dataset = YOLOConditionalDataset(
        images_dir=val_images_dir,
        labels_dir=val_labels_dir,
        image_size=image_size
    )
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    
    # Beta schedule
    betas = make_beta_schedule(start=1e-4, end=0.02, num_timesteps=num_timesteps).to(device)
    alphas = 1.0 - betas
    alphas_cumprod = torch.cumprod(alphas, dim=0)
    sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
    one_minus_alphas_cumprod = 1.0 - alphas_cumprod
    
    # Model, optimizer
    model = SimpleConditionalUNet(in_channels=6, base_channels=64).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    mse_loss = nn.MSELoss()
    
    # Track best validation accuracy
    best_val_acc = -1.0
    save_dir = "../runs/generative_models/DDPM/model/20epochs"
    os.makedirs(save_dir, exist_ok=True)
    
    # Training loop
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        train_acc = 0.0
        
        # Training
        train_loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{num_epochs}] [Train]", leave=True)
        for mask, real_img in train_loop:
            mask = mask.to(device)         # [B, 3, H, W]
            real_img = real_img.to(device) # [B, 3, H, W]
            
            # Sample a random t in [0, num_timesteps-1] for each example
            t = torch.randint(0, num_timesteps, (mask.size(0),), device=device).long()
            
            # Forward diffuse real_img to get x_t
            x_t, noise = forward_diffusion_sample(real_img, t, sqrt_alphas_cumprod, one_minus_alphas_cumprod)
            
            # Predict noise with our model
            pred_noise = model(x_t, mask)
            
            # Loss: L2 between predicted noise and true noise
            loss = mse_loss(pred_noise, noise)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            train_acc += 1 - loss.item()  # Placeholder for accuracy
            
            train_loop.set_postfix({
                "Train Loss": f"{train_loss / len(train_loader):.4f}",
                "Train Acc": f"{train_acc / len(train_loader):.4f}"
            })
        
        # Validation
        model.eval()
        val_loss = 0.0
        val_acc = 0.0
        
        with torch.no_grad():
            val_loop = tqdm(val_loader, desc=f"Epoch [{epoch+1}/{num_epochs}] [Val]", leave=True)
            for mask, real_img in val_loop:
                mask = mask.to(device)
                real_img = real_img.to(device)
                
                # Sample a random t
                t = torch.randint(0, num_timesteps, (mask.size(0),), device=device).long()
                
                # Forward diffuse real_img to get x_t
                x_t, noise = forward_diffusion_sample(real_img, t, sqrt_alphas_cumprod, one_minus_alphas_cumprod)
                
                # Predict noise
                pred_noise = model(x_t, mask)
                
                # Loss: L2 between predicted noise and true noise
                loss = mse_loss(pred_noise, noise)
                
                val_loss += loss.item()
                val_acc += 1 - loss.item()  # Placeholder for accuracy
                
                val_loop.set_postfix({
                    "Val Loss": f"{val_loss / len(val_loader):.4f}",
                    "Val Acc": f"{val_acc / len(val_loader):.4f}"
                })
        
        # Save last model
        last_model_path = os.path.join(save_dir, f"diffusion_model_last.pth")
        torch.save(model.state_dict(), last_model_path)
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_path = os.path.join(save_dir, f"diffusion_model_best.pth")
            torch.save(model.state_dict(), best_model_path)
            print(f"Best model saved with Val Acc: {best_val_acc:.4f}")
    
    print(f"Training complete. Last model saved to: {last_model_path}")

if __name__ == "__main__":
    train_diffusion()

### Synthesize

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as T
import cv2
from PIL import Image
import matplotlib.pyplot as plt

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"

class SimpleConditionalUNet(nn.Module):
    def __init__(self, in_channels=6, base_channels=64):
        """
        in_channels = 6 => (3 for noisy image + 3 for mask)
        out_channels = 3 => predict noise for the (RGB) image
        """
        super().__init__()
        self.enc1 = self.conv_block(in_channels, base_channels)
        self.enc2 = self.conv_block(base_channels, base_channels*2, down=True)
        self.enc3 = self.conv_block(base_channels*2, base_channels*4, down=True)
        
        self.middle = self.conv_block(base_channels*4, base_channels*4, down=False)
        
        self.dec3 = self.conv_block(base_channels*8, base_channels*2, up=True)
        self.dec2 = self.conv_block(base_channels*4, base_channels, up=True)
        self.dec1 = nn.Sequential(
            nn.Conv2d(base_channels*2, base_channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(base_channels, 3, kernel_size=1),  # Output = 3 channels
        )

    def conv_block(self, in_c, out_c, down=False, up=False):
        """
        down=True -> stride=2 conv
        up=True   -> stride=2 transpose conv
        else standard conv
        """
        if down:
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 4, 2, 1),
                nn.ReLU()
            )
        elif up:
            return nn.Sequential(
                nn.ConvTranspose2d(in_c, out_c, 4, 2, 1),
                nn.ReLU()
            )
        else:
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, 1, 1),
                nn.ReLU()
            )

    def forward(self, x, mask):
        """
        x : [B, 3, H, W] (noisy image)
        mask : [B, 3, H, W]
        """
        # Condition by concatenating
        c = torch.cat([x, mask], dim=1)  # [B, 6, H, W]
        e1 = self.enc1(c)               # [B, base, H, W]
        e2 = self.enc2(e1)              # [B, base*2, H/2, W/2]
        e3 = self.enc3(e2)              # [B, base*4, H/4, W/4]
        
        m = self.middle(e3)            # [B, base*4, H/4, W/4]
        
        d3 = self.dec3(torch.cat([m, e3], dim=1))  # skip conn
        d2 = self.dec2(torch.cat([d3, e2], dim=1))
        out = self.dec1(torch.cat([d2, e1], dim=1))
        return out

# Load the trained model
model = SimpleConditionalUNet(in_channels=6, base_channels=64).to(device)
checkpoint = "../runs/generative_models/DDPM/model/20epochs/diffusion_model_best.pth"
model.load_state_dict(torch.load(checkpoint, map_location=device))
model.eval()  # Set model to evaluation mode

# Beta schedule
def make_beta_schedule(start=1e-4, end=0.02, num_timesteps=1000):
    return torch.linspace(start, end, num_timesteps)  # Creates a linearly spaced schedule of betas.

num_timesteps = 200  # Match your code
betas = make_beta_schedule(start=1e-4, end=0.02, num_timesteps=num_timesteps).to(device)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)

def sample_diffusion(model, mask, betas, device):
    """
    Given a bounding-box mask, sample a random noise image and iteratively
    denoise it from T down to 0 using the model's predicted noise.
    
    mask: [1, 3, H, W]
    """
    model.eval()
    with torch.no_grad():
        num_timesteps = betas.shape[0]
        x = torch.randn_like(mask).to(device)  # x_T ~ N(0,I), shape = [1,3,H,W]

        alphas = 1.0 - betas
        alphas_cumprod = torch.cumprod(alphas, dim=0)
        
        for i in reversed(range(num_timesteps)):
            t = torch.tensor([i], device=device, dtype=torch.long)
            # Predict noise
            pred_noise = model(x, mask)
            
            # x_{t-1} = 1/sqrt(alpha_t) ( x_t - (1 - alpha_t)/sqrt(1 - alpha_cumprod_t) * pred_noise ) + ...
            alpha_t = alphas[t]
            alpha_cumprod_t = alphas_cumprod[t]
            beta_t = betas[t]
            
            coef1 = 1.0 / torch.sqrt(alpha_t)
            coef2 = (1.0 - alpha_t) / torch.sqrt(1.0 - alpha_cumprod_t)
            
            x = coef1 * ( x - coef2 * pred_noise )
            
            # Add noise, except at i=0
            if i > 0:
                z = torch.randn_like(x)
                sigma = torch.sqrt(beta_t)
                x = x + sigma * z
        
        return x  # x_0

# Load a test mask image
test_mask_path = "../data/generator_train/test/images/test-agg1-2067.png"
mask_img = cv2.imread(test_mask_path)
mask_img = cv2.cvtColor(mask_img, cv2.COLOR_BGR2RGB)
mask_img = cv2.resize(mask_img, (512, 384), interpolation=cv2.INTER_NEAREST)
mask_img = mask_img.astype(np.float32) / 255.0  # Normalize to [0, 1]
mask_tensor = torch.from_numpy(np.transpose(mask_img, (2, 0, 1))).unsqueeze(0).to(device)  # [1,3,H,W]

# Generate the image
x_0 = sample_diffusion(model, mask_tensor, betas, device)  # shape [1,3,H,W]

# Clamp the output to [0, 1] for visualization
x_0 = torch.clamp(x_0, 0.0, 1.0)

# Convert to numpy for visualization
out_np = x_0.squeeze(0).permute(1, 2, 0).cpu().numpy()  # [H,W,3]
gray = out_np.mean(axis=2, keepdims=True)  # shape (H,W,1)
out_np = np.repeat(gray, 3, axis=2)   # shape (H,W,3)

# 4) Check channel identity
identical_across_channels = (out_np[:,:,0] == out_np[:,:,1]).all() \
                            and (out_np[:,:,1] == out_np[:,:,2]).all()
if identical_across_channels:
    print("Now all channels are identical across each pixel.")
else:
    print("Something went wrong; channels differ.")

print('out_np.shape', out_np.shape)  # Should be [H, W, 3]
print('out_np min max values', np.min(out_np), np.max(out_np))

plt.imshow(out_np)
plt.axis("off")
plt.show()

### Generate

In [ ]:
import os
import shutil
import torch
import torch.nn as nn
import numpy as np
import cv2
from PIL import Image
import torchvision.transforms as T
import matplotlib.pyplot as plt

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------------------
# 1. Define the DDPM Model (from your training code)
# ------------------------------

class SimpleConditionalUNet(nn.Module):
    def __init__(self, in_channels=6, base_channels=64):
        """
        in_channels = 6 => (3 for noisy image + 3 for mask)
        out_channels = 3 => predict noise for the (RGB) image
        """
        super().__init__()
        self.enc1 = self.conv_block(in_channels, base_channels)
        self.enc2 = self.conv_block(base_channels, base_channels*2, down=True)
        self.enc3 = self.conv_block(base_channels*2, base_channels*4, down=True)
        
        self.middle = self.conv_block(base_channels*4, base_channels*4, down=False)
        
        self.dec3 = self.conv_block(base_channels*8, base_channels*2, up=True)
        self.dec2 = self.conv_block(base_channels*4, base_channels, up=True)
        self.dec1 = nn.Sequential(
            nn.Conv2d(base_channels*2, base_channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(base_channels, 3, kernel_size=1),  # Output = 3 channels
        )

    def conv_block(self, in_c, out_c, down=False, up=False):
        """
        down=True -> stride=2 conv
        up=True   -> stride=2 transpose conv
        else standard conv
        """
        if down:
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 4, 2, 1),
                nn.ReLU()
            )
        elif up:
            return nn.Sequential(
                nn.ConvTranspose2d(in_c, out_c, 4, 2, 1),
                nn.ReLU()
            )
        else:
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, 1, 1),
                nn.ReLU()
            )

    def forward(self, x, mask):
        """
        x : [B, 3, H, W] (noisy image)
        mask : [B, 3, H, W]
        """
        # Condition by concatenating
        c = torch.cat([x, mask], dim=1)  # [B, 6, H, W]
        e1 = self.enc1(c)               # [B, base, H, W]
        e2 = self.enc2(e1)              # [B, base*2, H/2, W/2]
        e3 = self.enc3(e2)              # [B, base*4, H/4, W/4]
        
        m = self.middle(e3)            # [B, base*4, H/4, W/4]
        
        d3 = self.dec3(torch.cat([m, e3], dim=1))  # skip conn
        d2 = self.dec2(torch.cat([d3, e2], dim=1))
        out = self.dec1(torch.cat([d2, e1], dim=1))
        return out

# ------------------------------
# 2. Load the Trained DDPM Model
# ------------------------------

def load_trained_ddpm_model(model_path, device):
    model = SimpleConditionalUNet(in_channels=6, base_channels=64).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()  # Set the model to evaluation mode
    return model

# ------------------------------
# 3. Reverse Diffusion (Sampling)
# ------------------------------

def reverse_diffusion_sample(model, mask, betas, device):
    """
    Generate a sample from the DDPM model using reverse diffusion.
    mask: [1, 3, H, W] (original image used as mask)
    """
    model.eval()
    with torch.no_grad():
        num_timesteps = betas.shape[0]
        x = torch.randn_like(mask).to(device)  # x_T ~ N(0,I), shape = [1,3,H,W]

        alphas = 1.0 - betas
        alphas_cumprod = torch.cumprod(alphas, dim=0)
        
        for i in reversed(range(num_timesteps)):
            t = torch.tensor([i], device=device, dtype=torch.long)
            # Predict noise
            pred_noise = model(x, mask)
            
            # x_{t-1} = 1/sqrt(alpha_t) ( x_t - (1 - alpha_t)/sqrt(1 - alpha_cumprod_t) * pred_noise ) + ...
            alpha_t = alphas[t]
            alpha_cumprod_t = alphas_cumprod[t]
            beta_t = betas[t]
            
            coef1 = 1.0 / torch.sqrt(alpha_t)
            coef2 = (1.0 - alpha_t) / torch.sqrt(1.0 - alpha_cumprod_t)
            
            x = coef1 * ( x - coef2 * pred_noise )
            
            # Add noise, except at i=0
            if i > 0:
                z = torch.randn_like(x)
                sigma = torch.sqrt(beta_t)
                x = x + sigma * z
        
        return x  # x_0

# ------------------------------
# 4. Synthesize Images with DDPM
# ------------------------------

def synthesize_images_ddpm(model, source_images_dir, target_images_dir, betas, image_size=(512, 384)):
    # Create target directory if it doesn't exist
    os.makedirs(target_images_dir, exist_ok=True)
    
    # Collect all image paths
    image_paths = sorted([os.path.join(source_images_dir, fname) for fname in os.listdir(source_images_dir) if fname.endswith('.png')])
    
    for img_path in image_paths:
        # Load the image
        test_image = cv2.imread(img_path)
        test_image = cv2.cvtColor(test_image, cv2.COLOR_BGR2RGB)  # Convert from BGR to RGB
        test_image = test_image.astype(np.float32) / 255.0  # Scale to [0, 1]
        
        # Resize the image to the desired size
        test_image = cv2.resize(test_image, image_size, interpolation=cv2.INTER_LINEAR)
        
        # Convert to tensor: [H, W, C] --> [C, H, W], then add batch dimension
        mask_tensor = torch.from_numpy(np.transpose(test_image, (2, 0, 1))).unsqueeze(0).to(device)
        
        # Perform reverse diffusion to generate the image
        synthesized_image_tensor = reverse_diffusion_sample(model, mask_tensor, betas, device)
        
        # Convert from [-1,1] to [0,1] for visualization
        synthesized_image_tensor = (synthesized_image_tensor + 1) / 2.0
        synthesized_image_tensor = torch.clamp(synthesized_image_tensor, 0, 1)
        
        # Convert to NumPy for saving
        synthesized_image_np = synthesized_image_tensor.squeeze(0).permute(1, 2, 0).cpu().numpy()  # [H, W, 3]
        gray = np.mean(synthesized_image_np, axis=2, keepdims=True)    # shape [H,W,1]
        synthesized_image_np = np.repeat(gray, 3, axis=2)              # shape [H,W,3]
        synthesized_image_np = (synthesized_image_np * 255).astype(np.uint8)  # Scale back to [0, 255]
        
        # Save the synthesized image
        target_image_path = os.path.join(target_images_dir, os.path.basename(img_path))
        cv2.imwrite(target_image_path, cv2.cvtColor(synthesized_image_np, cv2.COLOR_RGB2BGR))

# ------------------------------
# 5. Copy Images and Labels
# ------------------------------

def copy_images_and_labels(source_dir, target_dir):
    # Create target directory if it doesn't exist
    os.makedirs(target_dir, exist_ok=True)
    
    # Copy images
    for fname in os.listdir(source_dir):
        if fname.endswith('.png'):
            shutil.copy(os.path.join(source_dir, fname), os.path.join(target_dir, fname))
    
    # Copy labels
    labels_source_dir = source_dir.replace('images', 'labels')
    labels_target_dir = target_dir.replace('images', 'labels')
    os.makedirs(labels_target_dir, exist_ok=True)
    
    for fname in os.listdir(labels_source_dir):
        if fname.endswith('.txt'):
            label_path = os.path.join(labels_source_dir, fname)
            if os.path.exists(label_path):  # Check if the label file exists
                shutil.copy(label_path, os.path.join(labels_target_dir, fname))

# ------------------------------
# 6. Main Function to Generate Dataset
# ------------------------------

def generate_synthesized_dataset_ddpm():
    # Define base directories
    base_source_dir = "../data/agents"
    base_target_dir = "../data/synthetic_raw/DDPM"
    
    # List of agents
    agents = ['agent_0', 'agent_1', 'agent_2']
    
    # Load the trained DDPM model
    model_path = "../runs/generative_models/DDPM/model/20epochs/diffusion_model_best.pth"
    model = load_trained_ddpm_model(model_path, device)
    
    # Beta schedule
    num_timesteps = 200  # Match your training code
    betas = torch.linspace(1e-4, 0.02, num_timesteps).to(device)
    
    # Process each agent
    for agent in agents:
        # Define source and target directories for training data
        source_images_dir = os.path.join(base_source_dir, agent, 'train', 'images')
        target_images_dir = os.path.join(base_target_dir, f"{agent}_synthesized", 'train', 'images')
        
        # Synthesize images for the current agent
        synthesize_images_ddpm(model, source_images_dir, target_images_dir, betas)
        
        # Copy images and labels from the other two agents
        for other_agent in agents:
            if other_agent != agent:
                other_source_images_dir = os.path.join(base_source_dir, other_agent, 'train', 'images')
                copy_images_and_labels(other_source_images_dir, target_images_dir)
        
        # Copy labels for all agents
        for other_agent in agents:
            source_labels_dir = os.path.join(base_source_dir, other_agent, 'train', 'labels')
            target_labels_dir = os.path.join(base_target_dir, f"{agent}_synthesized", 'train', 'labels')
            copy_images_and_labels(source_labels_dir, target_labels_dir)
    
    # Repeat the process for the validation data
    for agent in agents:
        # Define source and target directories for validation data
        source_images_dir = os.path.join(base_source_dir, agent, 'valid', 'images')
        target_images_dir = os.path.join(base_target_dir, f"{agent}_synthesized", 'valid', 'images')
        
        # Synthesize images for the current agent
        synthesize_images_ddpm(model, source_images_dir, target_images_dir, betas)
        
        # Copy images and labels from the other two agents
        for other_agent in agents:
            if other_agent != agent:
                other_source_images_dir = os.path.join(base_source_dir, other_agent, 'valid', 'images')
                copy_images_and_labels(other_source_images_dir, target_images_dir)
        
        # Copy labels for all agents
        for other_agent in agents:
            source_labels_dir = os.path.join(base_source_dir, other_agent, 'valid', 'labels')
            target_labels_dir = os.path.join(base_target_dir, f"{agent}_synthesized", 'valid', 'labels')
            copy_images_and_labels(source_labels_dir, target_labels_dir)

    print("Dataset generation complete!")

# Run the dataset generation
if __name__ == "__main__":
    generate_synthesized_dataset_ddpm()
    # 1786m 37.1s

In [ ]:
import cv2

# Load the image
test_image_path = "../data/synthetic_raw/DDPM/agent_0_synthesized/train/images/train_b_cell_11_4_23_cap1_1011.png"
image = cv2.imread(test_image_path)

# Check if the image was loaded successfully
if image is None:
    print("Error: Unable to load image.")
else:
    # Get the number of channels
    num_channels = image.shape[2] if len(image.shape) == 3 else 1
    print(f"The image has {num_channels} channels.")

    # Check if all channels have the same values for each pixel
    if num_channels > 1:
        # Compare all channels
        is_identical = (image[:, :, 0] == image[:, :, 1]).all() and (image[:, :, 1] == image[:, :, 2]).all()
        if is_identical:
            print("All channels have the same values for each pixel.")
        else:
            print("The channels do not have the same values for each pixel.")
    else:
        print("The image is grayscale (only one channel).")